In [97]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import r2_score

In [ ]:
def read_data():
    train_path = '/Users/filippomontecchi/Desktop/Data Science & Machine Learning Lab/Lab/Dataset/LAB5/train_dataset.csv'
    test_path = '/Users/filippomontecchi/Desktop/Data Science & Machine Learning Lab/Lab/Dataset/LAB5/test_dataset.csv'
    
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    return train_df, test_df
    
train_df, test_df = read_data()
train_df.columns

- **SPLIT TRAIN DATASET INTO X_TRAIN, X_VAL, Y_TRAIN, Y_VAL** --> WORK ONLY ON X_TRAIN
- inspect % nan
- if there's dirty data (es. dirty text) clean to spot hidden nan / nan-like values
- drop clearly useless columns (id)
- histogram plot frequencies --> drop always constant columns
- manage nan DEPENDING ON THE PREPROCESSING: drop if % Nan is dramatic = >80% --> **otherwise impute / deal with Nan based on preprocess steps**

---

- **SPLIT TRAIN DATASET INTO X_TRAIN, X_VAL, Y_TRAIN, Y_VAL** --> WORK ONLY ON X_TRAIN

In [ ]:
data = train_df.loc[:, 'cont_0':'cat_7']
target = train_df['target']

x_train, x_val, y_train, y_val = train_test_split(data, target, test_size=0.2, random_state=42, shuffle=True)
x_train.shape

In [ ]:
count_na_col = x_train.isna().sum(axis=0).sort_values(ascending = False)
perc_na_col = x_train.isna().mean(axis=0).sort_values(ascending = False)


for col_name,count_na, perc_na in zip(count_na_col.index, count_na_col, perc_na_col):
    print(f'{col_name} : {count_na} --> {int(perc_na*100)}%')


- all columns have %Nan ≈18-21% --> completely normal == impute
- check columns

In [ ]:
print(x_train.isna().sum(axis=1).sort_values(ascending = False).head(5))
print()
print(x_train.isna().mean(axis=1).sort_values(ascending = False).head(5)*100)
print()
print(x_train.isna().mean(axis=1).mean()*100)


- worst rows have 24/58 Nan cols --> 41% Nan
- average number on nan is ≈20%

2 solutions:
- drop only rows with % Nan ≥ 40% (only 2 rows) 
- or keep everything and impute
    - I'll impute out of laziness

- search for dirty data (in ordinal and categorical columns)
    - search per column data type

In [33]:
num_col = x_train.loc[:, 'cont_0': 'cont_29']
ord_col = x_train.loc[:, 'ord_0': 'ord_19']
cat_col = x_train.loc[:, 'cat_0':'cat_7']

In [ ]:
all_ord = []

for o_c in ord_col:
    # print(ord_col[o_c].value_counts().head(5))
    # print(ord_col[o_c].value_counts().tail(10))
    non_nan_ord_values_per_col = ord_col[o_c].dropna()
    unique_ord_values = non_nan_ord_values_per_col.unique().tolist()
    sorted_unique_ord_values = sorted(unique_ord_values, key=lambda x:x.split('_')[3])
    all_ord.append(sorted_unique_ord_values)

# all_ord = np.unique(all_ord)[1:].tolist()
all_ord

- no werid values for the ordinal

> - I already know I'll perform an ordinal encoder on these ordinal columns --> I'll need to pass the list of categories otherwise if I use 'auto' it will sort them alphabetically.
> - save this list for later (removing Nans), please also note that not all columns have all these values specified --> you'll set encoded_missing_value to to specify how missing values should be handled --> they will be mapped to -2, while the nan values will be mapped to -1.

### THE ORDINAL ENCODER EXPECTS SOMETHING DIFFERENT
Like this:  
categories = [  
   ['cat_1', 'cat_2', 'cat_3'],  # for ord_0  
   ['cat_1', 'cat_2'],           # for ord_1  
   ['cat_1', 'cat_2', 'cat_3'],  # for ord_2  
   ... and so on for all ordinal columns  
]  

In [ ]:
all_cat = []

for c_c in cat_col:
    all_cat.extend(cat_col[c_c].unique().tolist())

print(np.unique(all_cat))

- no weird values except nan, which is expected, I'll deal with them later

- drop clearly useless columns (ID)
    - there's none

- plot frequencies
    - drop useless columns (only one value)
    - inspect frequencies based on col type

In [ ]:
plt.figure(figsize=(25,25))
for plt_idx, col in enumerate(num_col):
    plt.subplot(10,3,plt_idx+1)
    sns.histplot(x=num_col[col], bins='auto', kde=True)
    plt.title(f"Frequency plot of {col}")

plt.tight_layout()
plt.show()


- seems ok, no anomalies
- need to scale though

In [ ]:
plt.figure(figsize=(25,25))
for plt_idx, col in enumerate(ord_col):
    plt.subplot(10,2,plt_idx+1)
    sns.histplot(x=ord_col[col], bins='auto')
    plt.title(f"Frequency plot of {col}")

plt.tight_layout()
plt.show()

- seems ok, no one-value column
    - no dirty data

In [ ]:
plt.figure(figsize=(18,18))
for plt_idx, col in enumerate(cat_col):
    plt.subplot(4,2,plt_idx+1)
    sns.histplot(x=cat_col[col], bins='auto')
    plt.title(f"Frequency plot of {col}")

plt.tight_layout()
plt.show()

- drop categorical col cat_4, cat_5, cat_6 --> only one category = no additional info
- no dirty values

In [ ]:
# drop useless columns
x_train.drop(columns=['cat_4', 'cat_5', 'cat_6'], inplace=True)
x_train.shape

- MANAGE NANS BASED ON COLUMNS AND PREPROCESSING
    - numerical columns : need to scale --> impute Nan with simpleimputer median + missing col flag
    - ordinal columns : need to OrdinalEncode (passing list of ordinal values + specifing encoded_missing_value as -2) --> impute Nan with   simpliemputer constant 'unknown', then the ordinal encoder maps them to -1 with unknown_value=-1 + missing col flag  
###### - don't directly impute to -1 or the columntranformer won't be happy because the whole column has strings and out of no where there would be numbers (-1).
- categorical columns : need to OneHotEncode --> simpliemputer constant 'unknown' + missing col flag --> use 'unknown' so the OHE wil just build one column
- tie everything together with columntransformer

- try any BASE regression model --> randomforestregressor / linearregression / lasso / ridge / KNNregressor / logistic regressor
- pick the one with higher score WITHOUT TUNING, in the meantime tune eventual preprocessing hyperparameters --> then tune

In [ ]:
num_col = x_train.loc[:, 'cont_0': 'cont_29']
ord_col = x_train.loc[:, 'ord_0': 'ord_19']
cat_col = x_train.loc[:, 'cat_0':'cat_7']

num_col_names = ['cont_0', 'cont_1', 'cont_2', 'cont_3', 'cont_4', 'cont_5', 'cont_6',
       'cont_7', 'cont_8', 'cont_9', 'cont_10', 'cont_11', 'cont_12',
       'cont_13', 'cont_14', 'cont_15', 'cont_16', 'cont_17', 'cont_18',
       'cont_19', 'cont_20', 'cont_21', 'cont_22', 'cont_23', 'cont_24',
       'cont_25', 'cont_26', 'cont_27', 'cont_28', 'cont_29']
ord_col_names = ['ord_0', 'ord_1', 'ord_2', 'ord_3', 'ord_4', 'ord_5', 'ord_6', 'ord_7',
       'ord_8', 'ord_9', 'ord_10', 'ord_11', 'ord_12', 'ord_13', 'ord_14',
       'ord_15', 'ord_16', 'ord_17', 'ord_18', 'ord_19']
cat_col_names = ['cat_0', 'cat_1', 'cat_2', 'cat_3', 'cat_7']

In [ ]:
num_pipe = Pipeline(steps=[
    (
        'num_imputer',
        SimpleImputer(strategy='median', add_indicator=True)
    )
    ,
    (
        'scaler',
        StandardScaler()
    )
])

ord_pipe = Pipeline(steps=[
    (
        'ord_imputer',
        SimpleImputer(strategy='constant', fill_value='unknown')    # add_indicator = True --> if you explicitly pass categories it might fuck up everything
    )
    ,
    (
        'OE',
        OrdinalEncoder(categories=all_ord, handle_unknown='use_encoded_value', unknown_value=-1, encoded_missing_value=-2)      # Nan becomes -1 while missing ordinal values -2 : eg row 5 hasn't got ord_7_val_1 --> puts value -2
    )
])

cat_pipe = Pipeline(steps=[
    (
        'cat_imputer',
        SimpleImputer(strategy='constant', fill_value='unknown', add_indicator=True)
    )
])


preprocessing_pipeline = ColumnTransformer(transformers=[
    ('num_prep', num_pipe, num_col_names),
    ('ord_prep', ord_pipe, ord_col_names),
    ('cat_prep', cat_pipe, ord_col_names)
])

full_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessing_pipeline),
    ('regression', RandomForestRegressor())
])

full_pipeline.fit(x_train, y_train)

y_pred_train = full_pipeline.predict(x_train)
print(r2_score(y_train, y_pred_train))

y_pred = full_pipeline.predict(x_val)
print(r2_score(y_val, y_pred))



- linear regression:
    - 0.015369159498275109
    - -0.015871686097905346

- **RandomForestRegressor**  
    - 0.9676422853527189  
    - 0.7948775546560665

- Ridge
    - 0.015369158111415593
    - -0.015861601676245174

- Lasso
    - 0.0
    - -0.00014678128659451062

- KNeighborsRegressor
    - 0.7502138797840572
    - 0.6098325497257951

In [112]:
num_pipe = Pipeline(steps=[
    (
        'num_imputer',
        SimpleImputer(strategy='median', add_indicator=True)
    )
    ,
    (
        'scaler',
        StandardScaler()
    )
])

ord_pipe = Pipeline(steps=[
    (
        'ord_imputer',
        SimpleImputer(strategy='constant', fill_value='unknown')    # add_indicator = True --> if you explicitly pass categories it might fuck up everything
    )
    ,
    (
        'OE',
        OrdinalEncoder(categories=all_ord, handle_unknown='use_encoded_value', unknown_value=-1, encoded_missing_value=-2)      # Nan becomes -1 while missing ordinal values -2 : eg row 5 hasn't got ord_7_val_1 --> puts value -2
    )
])

cat_pipe = Pipeline(steps=[
    (
        'cat_imputer',
        SimpleImputer(strategy='constant', fill_value='unknown', add_indicator=True)
    )
    ,
    (
        'OHE',
        OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    )
])


preprocessing_pipeline = ColumnTransformer(transformers=[
    ('num_prep', num_pipe, num_col_names),
    ('ord_prep', ord_pipe, ord_col_names),
    ('cat_prep', cat_pipe, cat_col_names)
])

full_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessing_pipeline),
    ('regression', RandomForestRegressor(n_jobs=-1, random_state=42))
])

param_grid = {
    'regression__n_estimators': [300],
    'regression__max_depth' : [None],
    'regression__min_samples_split': [5],
    'regression__min_samples_leaf' : [5]
}

grid = GridSearchCV(estimator=full_pipeline, param_grid=param_grid, scoring='r2', cv=3, n_jobs=-1, verbose=1)
grid.fit(x_train, y_train)

print(grid.best_params_)
best_model = grid.best_estimator_

y_pred_train = best_model.predict(x_train)
print(r2_score(y_train, y_pred_train))

y_pred = best_model.predict(x_val)
print(r2_score(y_val, y_pred))

Fitting 3 folds for each of 1 candidates, totalling 3 fits
{'regression__max_depth': None, 'regression__min_samples_leaf': 5, 'regression__min_samples_split': 5, 'regression__n_estimators': 300}
0.9181602466059247
0.774771737988415


- eh good enough, we are on a time budget  
{'regression__max_depth': None, 'regression__min_samples_leaf': 5, 'regression__min_samples_split': 5, 'regression__n_estimators': 300}  
0.9181602466059247  
0.774771737988415  

In [109]:
def write_csv(y_pred):
    y_pred_df = pd.DataFrame({'ID': range(len(y_pred)), 'predictions': y_pred})
    y_pred_df.to_csv('predictions.csv', index=False)

write_csv(y_pred)

# MAIN.PY
- do not split into train and val il train_df --> from df_train build x_train and y_train ; from test x_test and y_test
- 
- remember to drop useless columns in the test set too